# ?? Rank #1 Leaderboard Strategy (Targeting 0.80+ Score)
This notebook applies **Column-Specific Target Optimization** and **LLM-as-a-Judge Clinical Post-Processing** on top of our fine-tuned NLLB-200 + Hybrid RAG pipeline.

In [ ]:
import os, sys, re, json, subprocess
from pathlib import Path
import pandas as pd, numpy as np, torch

try:
    _cwd = Path.cwd()
except (FileNotFoundError, OSError):
    os.chdir("/home/jovyan")
    _cwd = Path.cwd()

repo_name = "Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages"
if (_cwd / "src").exists(): BASE_DIR = _cwd
elif (_cwd.parent / "src").exists(): BASE_DIR = _cwd.parent
elif (_cwd / repo_name / "src").exists(): BASE_DIR = _cwd / repo_name
elif (Path("/home/jovyan") / repo_name / "src").exists(): BASE_DIR = Path("/home/jovyan") / repo_name
else: BASE_DIR = Path("/home/jovyan")

os.chdir(BASE_DIR)
for p in [str(BASE_DIR), str(BASE_DIR / "src")]:
    if p not in sys.path: sys.path.insert(0, p)

DATA_DIR        = BASE_DIR / "data" / "raw"
SUBMISSIONS_DIR = BASE_DIR / "submissions"
CHECKPOINTS_DIR = BASE_DIR / "models" / "checkpoints"
SRC_PATH        = BASE_DIR / "src" / "nllb_pipeline.py"

from retrieval import HybridRetriever
from nllb_pipeline import generate_nllb_answers
from advanced_ensemble import create_rank1_column_predictions
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(f"BASE_DIR: {BASE_DIR.resolve()}")
print(f"CUDA Available: {torch.cuda.is_available()}")


In [ ]:
train_df = pd.read_csv(DATA_DIR / "Training set.csv")
val_df   = pd.read_csv(DATA_DIR / "Validation set.csv")
test_df  = pd.read_csv(DATA_DIR / "Test set.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("[INFO] Initializing Hybrid RAG Retriever...")
retriever = HybridRetriever(train_df, enable_dense=True)

model_name = "facebook/nllb-200-1.3B"
ckpt_path = CHECKPOINTS_DIR / "nllb-1.3b-lora-checkpoint"
if not ckpt_path.exists():
    ckpt_path = CHECKPOINTS_DIR / "nllb-health-qa-checkpoint"

print(f"[INFO] Loading Model weights from {ckpt_path}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
try:
    from peft import PeftModel
    base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    model = PeftModel.from_pretrained(base_model, str(ckpt_path)).to(device)
    print("[INFO] Loaded LoRA PEFT adapter weights successfully!")
except Exception as e:
    print(f"[WARN] Loading base model ({e})...")
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

print("[INFO] Generating NLLB-200 generative predictions for Test Set...")
nllb_test_preds = generate_nllb_answers(model, tokenizer, test_df, retriever=retriever, device=device)

print("[INFO] Applying Rank #1 Column-Specific Target Optimization & LLM-Judge Post-Processing...")
rlf1_preds, r1f1_preds, llm_preds = create_rank1_column_predictions(test_df, nllb_test_preds, retriever=retriever)

sub_path = SUBMISSIONS_DIR / "submission_rank1_ensemble_0.8plus.csv"
sub_df = pd.DataFrame({
    "ID": test_df["ID"],
    "TargetRLF1": rlf1_preds,
    "TargetR1F1": r1f1_preds,
    "TargetLLM": llm_preds,
})
sub_df.to_csv(sub_path, index=False)
print(f"\n?? SUCCESS! Rank #1 Ensemble Submission saved to: {sub_path}")
display(sub_df.head(10))
